# 인덱스 복원 유틸리티

이 랩 전반에서 사용하는 미리 구성된 인덱스(`hrdocs` 및 `healthdocs`)를 복원합니다. 새로 고쳐진 노트북은 이 인덱스들이 `AZURE_SEARCH_SERVICE_ENDPOINT`로 구성된 검색 서비스에 이미 존재한다고 가정합니다.

In [ ]:
import json
import os
import traceback

from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.aio import SearchClient
from azure.search.documents.indexes.aio import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex

load_dotenv(override=True)
SEARCH_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
SEARCH_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]

credential = AzureKeyCredential(SEARCH_KEY)
print(f"Search endpoint : {SEARCH_ENDPOINT}")
print("Search auth     : api-key")
print(f"OpenAI endpoint : {AZURE_OPENAI_ENDPOINT}")

In [ ]:
async def restore_index(endpoint: str, credential, index_name: str, index_file: str, records_file: str, azure_openai_endpoint: str):
    default_path = r"../data/index-data"
    try:
        print(f"[{index_name}] Starting index restoration...")
        async with SearchIndexClient(endpoint=endpoint, credential=credential) as client:
            index_file_path = os.path.join(default_path, index_file)
            print(f"[{index_name}] Reading index definition from: {index_file_path}")
            with open(index_file_path, "r", encoding="utf-8") as in_file:
                index = SearchIndex._deserialize(json.load(in_file), True)
                index.name = index_name
                index.vector_search.vectorizers[0].parameters.resource_url = azure_openai_endpoint
                index.vector_search.vectorizers[0].parameters.api_key = AZURE_OPENAI_KEY
                await client.create_or_update_index(index)
        async with SearchClient(endpoint=endpoint, index_name=index_name, credential=credential) as client:
            records_file_path = os.path.join(default_path, records_file)
            print(f"[{index_name}] Reading documents from: {records_file_path}")
            records, total_uploaded, batch_count = [], 0, 0
            with open(records_file_path, "r", encoding="utf-8") as in_file:
                for line_num, line in enumerate(in_file, 1):
                    try:
                        records.append(json.loads(line))
                    except json.JSONDecodeError as e:
                        print(f"[{index_name}] WARNING: Skipping invalid JSON on line {line_num}: {e}")
                        continue
                    if len(records) >= 100:
                        batch_count += 1
                        print(f"[{index_name}] Uploading batch #{batch_count} ({len(records)} documents)...")
                        await client.upload_documents(documents=records)
                        total_uploaded += len(records)
                        records = []
            if records:
                batch_count += 1
                print(f"[{index_name}] Uploading final batch #{batch_count} ({len(records)} documents)...")
                await client.upload_documents(documents=records)
                total_uploaded += len(records)
        print(f"[{index_name}] SUCCESS - Index restored. Total documents uploaded: {total_uploaded}")
        return True
    except Exception as e:
        print(f"[{index_name}] ERROR - {type(e).__name__}: {e}")
        traceback.print_exc()
        return False

In [ ]:
print("\n--- Processing hrdocs index ---")
await restore_index(SEARCH_ENDPOINT, credential, "hrdocs", "index.json", "hrdocs-exported.jsonl", AZURE_OPENAI_ENDPOINT)

In [ ]:
print("\n--- Processing healthdocs index ---")
await restore_index(SEARCH_ENDPOINT, credential, "healthdocs", "index.json", "healthdocs-exported.jsonl", AZURE_OPENAI_ENDPOINT)

## 다음 단계

두 인덱스가 모두 성공적으로 복원되면, [1부: 복원된 검색 인덱스로 만드는 멀티 소스 KB](part1-standard-foundry-iq-kb.ipynb)부터 시작하세요.